# 🚀 Full Dataset Training on Kaggle GPU
## SVM + Bi-LSTM + BERT Models

**Optimized for Kaggle GPU with NO LIMITATIONS**

- Dataset: ~103K samples (72K train + 15.5K val + 15.5K test)
- GPU: Kaggle's P100 or T4 (16GB VRAM)
- Batch sizes: Maximized for GPU
- Epochs: Increased for better convergence
- All data processed fully without sampling

## 1️⃣ Import Libraries

In [1]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.2.0


In [2]:
!ls -l /kaggle/input/datasets/dx9029/mental-health
!ls -l /kaggle/working

total 310748
-rw-r--r-- 1 nobody nogroup       320 Mar 20 03:30 label_mapping.pkl
-rw-r--r-- 1 nobody nogroup  11342877 Mar 20 03:30 test_processed.csv
-rw-r--r-- 1 nobody nogroup  52998702 Mar 20 03:30 train_processed.csv
-rw-r--r-- 1 nobody nogroup  11316486 Mar 20 03:30 val_processed.csv
-rw-r--r-- 1 nobody nogroup  36247364 Mar 20 03:30 X_test_scaled.pkl
-rw-r--r-- 1 nobody nogroup 169219366 Mar 20 03:31 X_train_scaled.pkl
-rw-r--r-- 1 nobody nogroup  36252164 Mar 20 03:30 X_val_scaled.pkl
-rw-r--r-- 1 nobody nogroup    120985 Mar 20 03:30 y_test.pkl
-rw-r--r-- 1 nobody nogroup    564227 Mar 20 03:30 y_train.pkl
-rw-r--r-- 1 nobody nogroup    121001 Mar 20 03:30 y_val.pkl
total 24
---------- 1 root root 22177 Mar 20 03:31 __notebook__.ipynb


In [3]:
import pickle
from pathlib import Path
import warnings
import os
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Sklearn
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

warnings.filterwarnings('ignore')

# Check GPU availability
print("="*80)
print("SYSTEM CONFIGURATION")
print("="*80)
print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Name: {torch.cuda.get_device_name()}")
    print(f"✓ CUDA Version: {torch.version.cuda}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"✓ cuDNN Version: {torch.backends.cudnn.version()}")
print(f"✓ PyTorch Version: {torch.__version__}")
print()

SYSTEM CONFIGURATION
✓ GPU Available: True
✓ GPU Name: Tesla P100-PCIE-16GB
✓ CUDA Version: 12.6
✓ GPU Memory: 17.1 GB
✓ cuDNN Version: 91002
✓ PyTorch Version: 2.9.0+cu126



## 2️⃣ Setup Paths & Load Data

In [4]:
# ===== SETUP PATHS =====
# For Kaggle: mount data from dataset
data_dir = Path('/kaggle/input/datasets/dx9029/mental-health')  # Change to your dataset ID
output_dir = Path('/kaggle/working')

# Create output directory if needed
output_dir.mkdir(exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")
print()

# ===== LOAD FEATURES =====
print("="*80)
print("LOADING PREPROCESSED FEATURES")
print("="*80)

try:
    # Load scaled features
    with open(data_dir / "X_train_scaled.pkl", 'rb') as f:
        X_train = pickle.load(f)
    with open(data_dir / "X_val_scaled.pkl", 'rb') as f:
        X_val = pickle.load(f)
    with open(data_dir / "X_test_scaled.pkl", 'rb') as f:
        X_test = pickle.load(f)
    
    # Load labels
    with open(data_dir / "y_train.pkl", 'rb') as f:
        y_train = pickle.load(f)
    with open(data_dir / "y_val.pkl", 'rb') as f:
        y_val = pickle.load(f)
    with open(data_dir / "y_test.pkl", 'rb') as f:
        y_test = pickle.load(f)
    
    # Load label mapping
    with open(data_dir / "label_mapping.pkl", 'rb') as f:
        label_mapping = pickle.load(f)
    
    print(f"✓ X_train: {X_train.shape}")
    print(f"✓ X_val: {X_val.shape}")
    print(f"✓ X_test: {X_test.shape}")
    print(f"✓ y_train: {y_train.shape}")
    print(f"✓ y_val: {y_val.shape}")
    print(f"✓ y_test: {y_test.shape}")
    print(f"✓ Classes: {len(np.unique(y_train))}")
    print(f"✓ Label Mapping: {label_mapping}")
    print()
    
    num_classes = len(np.unique(y_train))
    
except Exception as e:
    print(f"❌ Error loading features: {e}")
    raise

Data directory: /kaggle/input/datasets/dx9029/mental-health
Output directory: /kaggle/working

LOADING PREPROCESSED FEATURES
✓ X_train: (70508, 300)
✓ X_val: (15105, 300)
✓ X_test: (15103, 300)
✓ y_train: (70508,)
✓ y_val: (15105,)
✓ y_test: (15103,)
✓ Classes: 7
✓ Label Mapping: {'anxiety': np.int64(0), 'bipolar': np.int64(1), 'depression': np.int64(2), 'normal': np.int64(3), 'personality disorder': np.int64(4), 'stress': np.int64(5), 'suicidal': np.int64(6)}



## 5️⃣ Model 1: SVM (Full Dataset)

In [5]:
print("="*80)
print("MODEL 1: LINEAR SVM - FULL DATASET")
print("="*80)
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Validation samples: {X_val.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")
print()

# Train SVM with NO limitations
print("▶ Training LinearSVC on FULL dataset...")
svm_model = LinearSVC(
    C=1.0,                    # Cost parameter (default, good for large datasets)
    max_iter=5000,            # ⬆️ Increased from 2000 for full convergence
    dual='auto',
    random_state=42,
    verbose=1,
    class_weight='balanced'   # Handle class imbalance
)
svm_model.fit(X_train, y_train)
print("✓ SVM training complete\n")

# Predict on all datasets
print("▶ Generating predictions...")
y_train_pred_svm = svm_model.predict(X_train)
y_val_pred_svm = svm_model.predict(X_val)
y_test_pred_svm = svm_model.predict(X_test)
print("✓ Predictions complete\n")

# Evaluate
print("-"*80)
print("SVM EVALUATION RESULTS")
print("-"*80)

svm_results = {
    'Train Accuracy': accuracy_score(y_train, y_train_pred_svm),
    'Val Accuracy': accuracy_score(y_val, y_val_pred_svm),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_svm),
    'Train Macro-F1': f1_score(y_train, y_train_pred_svm, average='macro'),
    'Val Macro-F1': f1_score(y_val, y_val_pred_svm, average='macro'),
    'Test Macro-F1': f1_score(y_test, y_test_pred_svm, average='macro'),
    'Train Weighted-F1': f1_score(y_train, y_train_pred_svm, average='weighted'),
    'Val Weighted-F1': f1_score(y_val, y_val_pred_svm, average='weighted'),
    'Test Weighted-F1': f1_score(y_test, y_test_pred_svm, average='weighted'),
}

for metric, value in svm_results.items():
    print(f"{metric:20s}: {value:.4f}")

# Classification report
print("\n" + "="*80)
print("SVM - Detailed Classification Report (Test Set)")
print("="*80)
print(classification_report(y_test, y_test_pred_svm, target_names=label_mapping.keys()))

# Save model
svm_file = output_dir / "svm_model_full.pkl"
with open(svm_file, 'wb') as f:
    pickle.dump(svm_model, f)
print(f"✓ SVM model saved to {svm_file}\n")

# Clean up memory
del svm_model
gc.collect()

MODEL 1: LINEAR SVM - FULL DATASET
Training samples: 70,508
Validation samples: 15,105
Test samples: 15,103

▶ Training LinearSVC on FULL dataset...
[LibLinear]iter  1 act 6.064e+04 pre 5.989e+04 delta 9.314e-01 f 6.829e+04 |g| 1.310e+05 CG   2
iter  2 act 1.550e+03 pre 1.208e+03 delta 9.314e-01 f 7.649e+03 |g| 1.077e+04 CG   3
iter  3 act 7.208e+02 pre 5.666e+02 delta 9.314e-01 f 6.099e+03 |g| 4.189e+03 CG   5
iter  4 act 2.913e+02 pre 2.334e+02 delta 9.314e-01 f 5.379e+03 |g| 1.682e+03 CG   7
iter  5 act 1.017e+02 pre 8.321e+01 delta 9.314e-01 f 5.087e+03 |g| 6.887e+02 CG  11
iter  6 act 3.318e+01 pre 2.824e+01 delta 9.314e-01 f 4.986e+03 |g| 2.855e+02 CG  20
iter  7 act 6.398e+00 pre 5.603e+00 delta 9.314e-01 f 4.952e+03 |g| 1.377e+02 CG  25
iter  8 act 5.324e-01 pre 5.159e-01 delta 9.314e-01 f 4.946e+03 |g| 4.220e+01 CG  26
iter  9 act 8.509e-03 pre 8.508e-03 delta 9.314e-01 f 4.945e+03 |g| 6.947e+00 CG  29
iter  1 act 6.438e+04 pre 6.368e+04 delta 9.551e-01 f 7.099e+04 |g| 1.357e+

255

## 5️⃣ Model 2: Bi-LSTM (Full Dataset + GPU)

In [6]:
# ===== BI-LSTM CLASS DEFINITION =====
class BiLSTMClassifier(nn.Module):
    """Bidirectional LSTM Classifier - Optimized for Large Datasets."""
    
    def __init__(self, input_size, hidden_size=256, num_layers=3, num_classes=7, dropout=0.4):
        super(BiLSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Bidirectional LSTM (INCREASED capacity for full dataset)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,        # ⬆️ Increased from 128 to 256
            num_layers=num_layers,          # ⬆️ Increased from 2 to 3
            batch_first=True,
            bidirectional=True,
            dropout=dropout                 # ⬆️ Increased from 0.3 to 0.4
        )
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Take last output
        output = self.fc(lstm_out)
        return output


print("="*80)
print("MODEL 2: BIDIRECTIONAL LSTM - FULL DATASET + GPU")
print("="*80)
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Feature dimension: {X_train.shape[1]}")
print()

# Prepare data for PyTorch
print("▶ Converting data to PyTorch tensors...")
X_train_tensor = torch.FloatTensor(X_train.toarray() if hasattr(X_train, 'toarray') else X_train)
X_val_tensor = torch.FloatTensor(X_val.toarray() if hasattr(X_val, 'toarray') else X_val)
X_test_tensor = torch.FloatTensor(X_test.toarray() if hasattr(X_test, 'toarray') else X_test)

y_train_tensor = torch.LongTensor(y_train)
y_val_tensor = torch.LongTensor(y_val)
y_test_tensor = torch.LongTensor(y_test)

# Add sequence dimension (required for LSTM)
X_train_tensor = X_train_tensor.unsqueeze(1)
X_val_tensor = X_val_tensor.unsqueeze(1)
X_test_tensor = X_test_tensor.unsqueeze(1)

print(f"   ✓ X_train_tensor: {X_train_tensor.shape}")
print(f"   ✓ X_val_tensor: {X_val_tensor.shape}")
print(f"   ✓ X_test_tensor: {X_test_tensor.shape}")
print()

# Create DataLoaders with MAXIMIZED batch sizes for GPU
print("▶ Creating DataLoaders with MAXIMUM batch sizes...")
batch_size_train = 128  # ⬆️ Increased from 32 to 128 (GPU can handle)
batch_size_val = 128

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size_val, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size_val, shuffle=False, num_workers=2)

print(f"   ✓ Train loader: {len(train_loader)} batches")
print(f"   ✓ Val loader: {len(val_loader)} batches")
print(f"   ✓ Batch size: {batch_size_train}")
print()

# Build model
print("▶ Building Bi-LSTM model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = BiLSTMClassifier(
    input_size=X_train_tensor.shape[2],
    hidden_size=256,        # ⬆️ Increased
    num_layers=3,           # ⬆️ Increased
    num_classes=num_classes,
    dropout=0.4             # ⬆️ Increased
).to(device)

print(f"   ✓ Model moved to {device}")
print(f"   ✓ Total parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")
print()

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(lstm_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)  # Better learning rate schedule

print("▶ Starting Bi-LSTM training...")
num_epochs = 10  # Changed from 50 to 10 epochs
best_val_acc = 0
patience = 10    # ⬆️ Increased from 5 (more patience for complex dataset)
patience_counter = 0
best_lstm_state = None

train_losses = []
val_accs = []

for epoch in tqdm(range(num_epochs), desc="Training LSTM"):
    # Training phase
    lstm_model.train()
    train_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = lstm_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    lstm_model.eval()
    val_correct = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = lstm_model(X_batch)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == y_batch).sum().item()
    
    val_acc = val_correct / len(y_val)
    val_accs.append(val_acc)
    scheduler.step()
    
    # Early stopping logic
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_lstm_state = lstm_model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}, Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\n✓ Bi-LSTM training complete")
print(f"✓ Best validation accuracy: {best_val_acc:.4f}")

# Load best model
lstm_model.load_state_dict(best_lstm_state)

# Generate predictions
print("\n▶ Generating predictions on all sets...")
lstm_model.eval()
with torch.no_grad():
    y_train_pred_lstm = lstm_model(X_train_tensor.to(device)).argmax(1).cpu().numpy()
    y_val_pred_lstm = lstm_model(X_val_tensor.to(device)).argmax(1).cpu().numpy()
    y_test_pred_lstm = lstm_model(X_test_tensor.to(device)).argmax(1).cpu().numpy()

print("✓ Predictions complete\n")

# Evaluate
print("-"*80)
print("BI-LSTM EVALUATION RESULTS")
print("-"*80)

lstm_results = {
    'Train Accuracy': accuracy_score(y_train, y_train_pred_lstm),
    'Val Accuracy': accuracy_score(y_val, y_val_pred_lstm),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_lstm),
    'Train Macro-F1': f1_score(y_train, y_train_pred_lstm, average='macro'),
    'Val Macro-F1': f1_score(y_val, y_val_pred_lstm, average='macro'),
    'Test Macro-F1': f1_score(y_test, y_test_pred_lstm, average='macro'),
    'Train Weighted-F1': f1_score(y_train, y_train_pred_lstm, average='weighted'),
    'Val Weighted-F1': f1_score(y_val, y_val_pred_lstm, average='weighted'),
    'Test Weighted-F1': f1_score(y_test, y_test_pred_lstm, average='weighted'),
}

for metric, value in lstm_results.items():
    print(f"{metric:20s}: {value:.4f}")

# Classification report
print("\n" + "="*80)
print("BI-LSTM - Detailed Classification Report (Test Set)")
print("="*80)
print(classification_report(y_test, y_test_pred_lstm, target_names=label_mapping.keys()))

# Save model
lstm_file = output_dir / "lstm_model_full.pt"
torch.save(lstm_model.state_dict(), lstm_file)
print(f"✓ LSTM model saved to {lstm_file}\n")

# Clean up
del lstm_model, train_loader, val_loader, test_loader
del X_train_tensor, X_val_tensor, X_test_tensor
gc.collect()
torch.cuda.empty_cache()

MODEL 2: BIDIRECTIONAL LSTM - FULL DATASET + GPU
Training samples: 70,508
Feature dimension: 300

▶ Converting data to PyTorch tensors...
   ✓ X_train_tensor: torch.Size([70508, 1, 300])
   ✓ X_val_tensor: torch.Size([15105, 1, 300])
   ✓ X_test_tensor: torch.Size([15103, 1, 300])

▶ Creating DataLoaders with MAXIMUM batch sizes...
   ✓ Train loader: 551 batches
   ✓ Val loader: 119 batches
   ✓ Batch size: 128

▶ Building Bi-LSTM model...
   ✓ Model moved to cuda
   ✓ Total parameters: 4,692,487

▶ Starting Bi-LSTM training...


Training LSTM:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 10/10 - Loss: 0.1182, Val Acc: 0.8540, Patience: 6/10

✓ Bi-LSTM training complete
✓ Best validation accuracy: 0.8575

▶ Generating predictions on all sets...
✓ Predictions complete

--------------------------------------------------------------------------------
BI-LSTM EVALUATION RESULTS
--------------------------------------------------------------------------------
Train Accuracy      : 0.9734
Val Accuracy        : 0.8540
Test Accuracy       : 0.8607
Train Macro-F1      : 0.9716
Val Macro-F1        : 0.8490
Test Macro-F1       : 0.8557
Train Weighted-F1   : 0.9734
Val Weighted-F1     : 0.8547
Test Weighted-F1    : 0.8612

BI-LSTM - Detailed Classification Report (Test Set)
                      precision    recall  f1-score   support

             anxiety       0.94      0.94      0.94      2629
             bipolar       0.96      0.93      0.95      2055
          depression       0.70      0.67      0.69      2376
              normal       0.86      0.90      0.88      20